# Tool 1: RAG

In [0]:

spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.rag_search(
    question STRING COMMENT 'The medical facility question to search semantically',
    region   STRING COMMENT 'Optional Ghana region filter e.g. Greater Accra Region'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Semantic search over Ghana medical facility documents using RAG vector search'
AS $$
import requests
import json

WORKSPACE_URL = "your-WORKSPACE_URL"
RAG_ENDPOINT  = "your-RAG_ENDPOINT"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

payload = {
    "dataframe_records": [
        {
            "question": question, 
            "region"  : region if region else None
        }
    ]
}

try:
    resp = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/{RAG_ENDPOINT}/invocations",
        headers = headers,
        json    = payload,
        timeout = 180
    )
    if resp.status_code != 200:
        return json.dumps({"error": f"RAG failed: {resp.status_code}: {resp.text}"})

    pred      = resp.json()["predictions"][0]
    citations = pred.get("citations", "[]")
    mappable  = pred.get("mappable_facilities", "[]")

    return json.dumps({
        "answer"     : pred.get("answer", ""),
        "citations"  : json.loads(citations) if isinstance(citations, str) else citations,
        "mappable"   : json.loads(mappable)  if isinstance(mappable,  str) else mappable,
        "num_sources": pred.get("num_sources", 0)
    })

except Exception as e:
    return json.dumps({"error": str(e)})
$$
""")

print("rag_search UC Function created successfully")